## Loading Base Model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

# ===== 1. 环境验证 =====
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ===== 2. 配置 4-bit 量化 =====
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# ===== 3. 加载基座模型和 tokenizer =====
model_name = "Qwen/Qwen3-1.7B-Base"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

special_tokens = {
    "additional_special_tokens": [
        "<tools>",
        "</tools>",
        "<tool_calls>",
        "</tool_calls>",
        "<tool_results>",
        "</tool_results>",
        "<tool_response>",
        "</tool_response>",
    ]
}

tokenizer.add_special_tokens(special_tokens)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print(f"Model loaded. Parameters: {model.num_parameters() / 1e6:.1f}M")
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

# ===== 4. 配置 LoRA 适配器 =====
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

PyTorch version: 2.8.0+cu128
CUDA available: True
GPU: NVIDIA A800-SXM4-80GB
GPU Memory: 85.1 GB


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Model loaded. Parameters: 1720.6M
Memory footprint: 1.33 GB
trainable params: 34,865,152 || all params: 1,755,440,128 || trainable%: 1.9861


## Data Preprocessing

In [ ]:
import datasets
import re, json
from datasets import Dataset, load_dataset, load_from_disk


tool_ds = load_from_disk("./tool_subset").shuffle(seed=42).select(range(4000))
print(f"Selected samples: {len(tool_ds)}")


Selected samples: 4000


In [1]:
# list of dict -> list of tools:{name, description, parameters} if role == 'assistant' and functions is not None
def extract_functions(functions_str)-> list:
    data = json.loads(functions_str)   # 先转成 list[dict]
    return [f["function"] for f in data]

# list of dict, each dict has keys: tool_name, tool_results if role is environment
def parse_tool_result(content)-> list:
    results = []

    for line in content.split("\n"):
        line = line.strip()
        if line:
            try:
                results.append(json.loads(line))
            except:
                results.append(line)  # fallback

    return results

def parse_tool_calls(text):

    calls = []

    # 按行拆
    lines = [l.strip() for l in text.split("\n") if l.strip()]

    for line in lines:
        # 匹配: func(arg=val, ...)
        match = re.match(r"(\w+(?:\.\w+)*)\((.*)\)", line)
        if not match:
            continue

        func_name = match.group(1)
        args_str = match.group(2)

        args = {}

        # 匹配 key="value" 或 key=123
        for k, v in re.findall(r"(\w+)\s*=\s*('[^']*'|\"[^\"]*\"|\d+)", args_str):
            v = v.strip()

            # 去引号
            if v.startswith(("'", '"')) and v.endswith(("'", '"')):
                v = v[1:-1]
            else:
                # 转 int
                if v.isdigit():
                    v = int(v)

            args[k] = v

        calls.append({
            "name": func_name,
            "arguments": args
        })

    return calls

In [2]:
def process_message(example):
    messages = example["messages"]
    new_messages = []
    for i in messages:
        # tool_calls
        if i['role'] == 'assistant' and i['function_calls']:
            function_calls = parse_tool_calls(i['function_calls'])
            new_messages.append({
                'role': i['role'],
                "content": f"<tool_calls>{json.dumps(function_calls, ensure_ascii=False)}</tool_calls>"
            })
        # assistant content
        elif i['role'] == 'assistant' and i['content']:
            new_messages.append({
                'role': i['role'],
                'content': i['content']
            })
        # system prompt with tools
        if i['role'] == 'system' and i['functions']:
            functions = extract_functions(i['functions'])
            new_messages.append({
                'role': 'system',
                'content':f"""
                    You are a helpful function-calling AI assistant.
                    You are provided with function signatures within <tools></tools> XML tags.
                    When needed, call tools by outputting tool calls in <tool_calls></tool_calls> XML tags.
                    Do not include any other text when making tool calls.
                    <tools>
                    {json.dumps(functions, ensure_ascii=False)}
                    </tools>
                """
            })
        # tool results
        if i['role'] == 'environment' and i['content']:
            result = parse_tool_result(i['content'])
            new_messages.append({
                'role': 'tool',
                'content': f"""
                <tool_results>
                {json.dumps(result, ensure_ascii=False)}
                </tool_results>
                """
            })
        # user content
        if i['role'] == 'user' and i['content']:
            new_messages.append({
                'role': i['role'],
                'content': i['content']
            })
    return {'messages': new_messages}

In [ ]:
tool_ds = tool_ds.map(process_message, num_proc=4)

In [ ]:
chat_ds = datasets.load_from_disk("./chat_subset").shuffle(seed=42).select(range(4000))

In [10]:
# 检查当前列，只删除存在的多余列
cols_to_remove = [c for c in tool_ds.column_names if c != "messages"]
if cols_to_remove:
    tool_ds = tool_ds.remove_columns(cols_to_remove)
cols_to_remove = [c for c in chat_ds.column_names if c != "messages"]
if cols_to_remove:
    chat_ds = chat_ds.remove_columns(cols_to_remove)

In [ ]:
# ===== 3. 格式化为 ChatML =====
def format_chat(example):
    """将消息列表转换为 ChatML 格式的文本"""
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

tool_ds = tool_ds.map(format_chat)
chat_ds = chat_ds.map(format_chat)

In [12]:
tool_ds = tool_ds.remove_columns(
    [c for c in tool_ds.column_names if c != "text"]
)

chat_ds = chat_ds.remove_columns(
    [c for c in chat_ds.column_names if c != "text"]
)

dataset = datasets.concatenate_datasets([tool_ds, chat_ds])

In [ ]:
def filter_long(example):
    length = len(tokenizer(example["text"]).input_ids)
    return length < 4096   # 或 4096

dataset = dataset.filter(filter_long)

In [ ]:
def truncate(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=2048
    )
    example["text"] = tokenizer.decode(tokens["input_ids"])
    return example

dataset = dataset.map(truncate)

In [15]:
# ===== 5. 划分训练集和验证集 =====
split_dataset = dataset.shuffle(seed=42).train_test_split(test_size=0.05, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]
print(f"\nTrain: {len(train_dataset)}, Eval: {len(eval_dataset)}")


Train: 7198, Eval: 379


## SFT Training


In [16]:
from trl import SFTTrainer, SFTConfig

# ===== 1. 训练配置 =====
sft_config = SFTConfig(
    output_dir="./qwen3-1.7b-sft",

    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,

    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    max_length=2048,
    packing=False,

    logging_steps=10,
    eval_strategy="steps",
    eval_steps=20,
    save_strategy="steps",
    save_steps=400,
    save_total_limit=3,

    report_to="none",
    seed=42,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

In [18]:
lengths = [len(tokenizer(x["text"]).input_ids) for x in dataset]

print("avg:", sum(lengths)/len(lengths))
print("max:", max(lengths))

avg: 1133.2586775768773
max: 2048


In [ ]:
# ===== 3. 开始训练 =====
print("Starting training...")
train_result = trainer.train()

# ===== 4. 保存模型 =====
trainer.save_model("./qwen3-1.7b-sft/final")
tokenizer.save_pretrained("./qwen3-1.7b-sft/final")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting training...


Step,Training Loss,Validation Loss
20,1.339605,1.241717
40,1.197358,1.141263
60,1.159660,1.120032
80,1.143528,1.110169
100,1.111155,1.103129
120,1.143027,1.096853
140,1.102160,1.093335
160,1.107405,1.090229
180,1.116878,1.088569
200,1.101213,1.087982


In [ ]:

# ===== 5. 打印训练结果 =====
print(f"\nTraining completed!")
print(f"  Train loss: {train_result.training_loss:.4f}")
print(f"  Train runtime: {train_result.metrics['train_runtime']:.0f}s")
print(f"  Train samples/sec: {train_result.metrics['train_samples_per_second']:.2f}")

## Visiualization result

In [ ]:
import matplotlib.pyplot as plt

# 从 trainer 日志中提取损失值
log_history = trainer.state.log_history
train_losses = [(x["step"], x["loss"]) for x in log_history if "loss" in x]
eval_losses = [(x["step"], x["eval_loss"]) for x in log_history if "eval_loss" in x]

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(*zip(*train_losses), label="Train Loss", alpha=0.8)
if eval_losses:
    ax.plot(*zip(*eval_losses), label="Eval Loss", marker="o")
ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title("SFT Training Loss Curve")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("training_loss.png", dpi=150)
plt.show()
print("Loss curve saved to training_loss.png")